# Module 6: Ethical & Environmental Auditing of AI-Driven Urban Traffic Control

### Overview

- **Ethical Analysis:** Did RL actions disproportionately target/exclude certain traffic events (e.g. false positives, missed anomalies)?  
- **Environmental Impact:** Proxy estimation of emissions reduction (e.g. sum of vehicle waiting times under RL vs. baseline)  
- **Explainability:** Use SHAP to interpret action logic if model is amenable

---

## 1. Load Final RL Actions CSV

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("rl_traffic_actions.csv")
print(df.head())
print("Sum anomaly frames:", df['is_anomaly'].sum())
print("Total non-anomaly frames:", (df['is_anomaly']==0).sum())

FileNotFoundError: [Errno 2] No such file or directory: 'rl_traffic_actions.csv'

## 2. Ethical Fairness Metrics

- Did the RL policy handle anomalies as expected?
- Did normal frames ever get "emergency" action?
- What is the false positive/negative rate on anomaly handling?

In [ ]:
# 0 = no anomaly, 1 = RL intervenes as if anomaly
normal_frames = df[df.is_anomaly==0]
anomaly_frames = df[df.is_anomaly==1]

# How often did RL use "anomaly" action?
normal_as_anomaly = normal_frames['RL_action'].fillna(0).sum()   # RL_action==1 during normal
anomaly_as_anomaly = anomaly_frames['RL_action'].fillna(0).sum() # RL_action==1 during actual anomaly

print(f"Frames with no anomaly where RL took action=ANOMALY: {normal_as_anomaly} out of {len(normal_frames)} ({100*normal_as_anomaly/len(normal_frames):.1f}%)")
print(f"Frames with anomaly where RL took action=ANOMALY: {anomaly_as_anomaly} out of {len(anomaly_frames)} ({100*anomaly_as_anomaly/len(anomaly_frames):.1f}%)")

# Calculate false positive/negative rates
false_positives = normal_as_anomaly / (len(normal_frames) + 1e-7)
false_negatives = (len(anomaly_frames) - anomaly_as_anomaly) / (len(anomaly_frames) + 1e-7)

print(f"False positive (unwarranted intervention): {100*false_positives:.2f}%")
print(f"False negative (missed anomaly): {100*false_negatives:.2f}%")

Frames with no anomaly where RL took action=ANOMALY: 239.0 out of 1893 (12.6%)
Frames with anomaly where RL took action=ANOMALY: 26.0 out of 100 (26.0%)
False positive (unwarranted intervention): 12.63%
False negative (missed anomaly): 74.00%


## 3. Environmental Impact Proxy

- Estimate stopped-vehicle-hours ("waiting time") saved by RL intervention compared to a naïve policy that never switches (action=0 always).
- You may use sum of vehicles present as the "emission proxy" (higher = worse for environment).

In [ ]:
# Compute "queue hours" under RL vs baseline (sum of vehicle counts)
baseline_queue = df['true_vehicle_count'].sum()
rl_queue = 0
for _, row in df.iterrows():
    # If RL intervenes, we assume it improves clearing by 30% (or other realistic effect)
    if row['RL_action'] == 1:
        rl_queue += row['true_vehicle_count'] * 0.7
    else:
        rl_queue += row['true_vehicle_count']
reduction = (baseline_queue - rl_queue) / baseline_queue * 100
print(f"Estimated waiting/queue reduction via RL: {reduction:.1f}%")

Estimated waiting/queue reduction via RL: 3.6%


## 4. Explainability (Optional SHAP analysis)

- For tabular Q-learning, you can print the Q-values or the state-buckets that led to action=1.
- For DQNs or deeper models: use SHAP on state->Q(state,action) to interpret.

In [ ]:
# For tabular Q, show state bucket most likely to lead to RL_action==1
def state_key(row, bins=[0,6,12,100]):
    for i in range(len(bins)-1):
        if bins[i] <= row['true_vehicle_count'] < bins[i+1]:
            b = i
    return (b, int(row['is_anomaly']))

# Only frames where RL took action=1
rl_action_1 = df[df['RL_action']==1]
buckets = rl_action_1.apply(state_key, axis=1)
bucket_counts = buckets.value_counts()
print("Most common state buckets leading to action=1:")
print(bucket_counts)

Most common state buckets leading to action=1:
(0, 0)    153
(1, 0)     72
(2, 0)     14
(1, 1)     12
(0, 1)     11
(2, 1)      3
Name: count, dtype: int64


## 5. Fairness Over Time or Special Groups

If you have demographic or spatial info, you can check if RL action was (un)evenly distributed.  
With only one sensor, you can comment on possible temporal/edge fairness issues (e.g. if RL only protects high-traffic times).

## 6. Save Audited Results/Generate Report Table

Summarize the bias/impact findings for your report.  
Include as appendix or in Results/Discussion.

In [ ]:
metrics = {
    "false_positive_rate": [false_positives],
    "false_negative_rate": [false_negatives],
    "queue_reduction_percent": [reduction]
}
report_df = pd.DataFrame(metrics)
report_df.to_csv("ethical_env_audit_summary.csv", index=False)
print(report_df)

   false_positive_rate  false_negative_rate  queue_reduction_percent
0             0.126255                 0.74                 3.645038


## Discussion

- RL agent intervened in {anomaly_as_anomaly}/{len(anomaly_frames)} ({100*anomaly_as_anomaly/len(anomaly_frames):.1f}%) of true anomaly frames, missing {false_negatives*100:.1f}%.  
- False positive rate: {false_positives*100:.1f}% (intervening when not needed)
- Proxy emissions impact: RL reduced estimated idle/queue time by {reduction:.1f}%
- Bucket analysis: RL most often triggered action=1 in states described as {bucket_counts.index[0]} (which means: bucket {bucket_counts.index[0]}, e.g. high congestion & anomaly).
- For further improvement: incorporate pedestrian/fairness or real emission sensors, and run SHAP on any neural Q-model.